# Kubernetes

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/kubernetes/kubernetes.ipynb)

Kubernetes is a container orchestrator: you declare desired state (run four replicas of this image with these resources), and controllers continuously reconcile the cluster toward it. Data platforms use it for Spark, Airflow, Flink, Trino and batch jobs. This notebook covers the objects, scheduling, resources, storage, networking, security, autoscaling, batch schedulers (Kueue, Volcano, YuniKorn) and debugging that matter for data workloads. Manifests are shown as YAML; scheduling, QoS, autoscaling and CronJob behaviour are simulated in Python, so nothing calls `kubectl` or a cluster.

*Last reviewed: November 2026, against Kubernetes 1.37.*

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"yaml": "pyyaml>=6.0"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## Why Kubernetes for data engineering

+ **Isolation and reproducibility**: every task runs in its own image with pinned dependencies, so one team's library upgrade cannot break another's DAG.
+ **Elastic compute**: executors and workers scale with the queue, and node autoscaling (Cluster Autoscaler, Karpenter) adds and removes machines, including cheap spot capacity.
+ **One platform** for batch (Jobs, Spark), streaming (Flink, Kafka via operators), services (Trino, Superset, APIs) and orchestration (Airflow).
+ The costs: operational complexity, networking and storage expertise, and the temptation to run stateful databases yourself that a managed service would run better.

## Architecture

**Control plane**

+ `kube-apiserver`: the only component that talks to etcd; every client (kubectl, controllers, kubelets) goes through its REST API, with authentication, RBAC authorization and admission control.
+ `etcd`: consistent key-value store holding all cluster state. Back it up.
+ `kube-scheduler`: assigns pending pods to nodes (filter feasible nodes, then score them).
+ `kube-controller-manager`: reconciliation loops (Deployment, ReplicaSet, Job, node lifecycle, endpoints).
+ `cloud-controller-manager`: cloud integration (load balancers, node lifecycle, routes).

**Every node**

+ `kubelet`: starts pods via the container runtime, runs probes, reports status, evicts pods under resource pressure.
+ Container runtime: containerd or CRI-O (images built with Docker run unchanged).
+ `kube-proxy` (or an eBPF CNI such as Cilium): implements Service virtual IPs.
+ CNI plugin: pod networking (every pod gets a routable IP).

Managed offerings (GKE, EKS, AKS) run the control plane for you; GKE Autopilot also manages nodes and bills per pod request. Locally, use `kind`, `minikube` or `k3d`.

## Workload objects

| Object | Semantics | Data use |
|---|---|---|
| Pod | One or more containers sharing network and volumes; the scheduling unit | Rarely created directly |
| Deployment | Stateless replicas, rolling updates, replaced pods get new names | Airflow webserver, Superset, APIs, Trino workers |
| StatefulSet | Stable names (`kafka-0`), ordered rollout, per-replica PVC via `volumeClaimTemplates` | Kafka, ZooKeeper, databases, Airflow Celery workers that need stable logs |
| DaemonSet | One pod per node | Log shippers, node exporters, CSI drivers |
| Job | Runs pods to completion with retries (`backoffLimit`), parallelism and completions | dbt runs, backfills, one-off migrations |
| CronJob | Creates Jobs on a cron schedule | Scheduled extracts and maintenance |

A Deployment controls a ReplicaSet, which keeps N pods matching a label selector. Rolling updates replace pods gradually, bounded by `maxSurge` (extra pods allowed) and `maxUnavailable` (pods allowed to be missing):

```yaml
apiVersion: apps/v1
kind: Deployment
metadata: {name: trino-worker, namespace: analytics}
spec:
  replicas: 4
  strategy:
    type: RollingUpdate
    rollingUpdate: {maxSurge: 25%, maxUnavailable: 0}
  selector: {matchLabels: {app: trino-worker}}
  template:
    metadata: {labels: {app: trino-worker}}
    spec:
      containers:
        - name: trino
          image: trinodb/trino:463
          resources:
            requests: {cpu: "4", memory: 16Gi}
            limits: {memory: 16Gi}
```

## Resources: requests, limits and QoS

+ **Requests** are what the scheduler reserves on a node. A pod fits only if the sum of requests on the node stays within its allocatable capacity, regardless of actual usage.
+ **Limits** are enforced by the kernel: exceeding a **CPU** limit throttles the container (CFS quota); exceeding a **memory** limit gets it OOM-killed (exit code 137, reason `OOMKilled`).
+ Units: CPU in cores or millicores (`500m` = half a core); memory in bytes with binary (`Gi`) or decimal (`G`) suffixes.

The QoS class follows from requests and limits and decides eviction order under node memory pressure:

| QoS | Rule | Evicted |
|---|---|---|
| Guaranteed | Every container has CPU and memory requests equal to limits | Last |
| Burstable | Something is set, but not Guaranteed | In between, by usage above request |
| BestEffort | Nothing set | First |

Practical advice: always set memory requests equal to memory limits for JVM and pandas workloads (memory is not compressible, so overcommit ends in OOM kills); set CPU requests from observed usage and consider no CPU limit, or a generous one, because CPU limits cause throttling and latency spikes even when the node is idle. `LimitRange` sets defaults per namespace; `ResourceQuota` caps a namespace's total requests, which is how you keep one team's backfill from starving everyone else.

### Where you'll use it

- **Running Spark and Flink on shared infrastructure:** Spark on Kubernetes and the Flink operator schedule data jobs next to other services.
- **Scheduled batch work:** CronJobs run nightly exports and compaction jobs without a dedicated VM.

### Where you'll use it

- **Stopping noisy neighbours:** a memory limit on a Spark executor stops one job from starving every other pod on the node.

In [ ]:
import re

UNITS = {"Ki": 2**10, "Mi": 2**20, "Gi": 2**30, "Ti": 2**40, "K": 10**3, "M": 10**6, "G": 10**9, "T": 10**12}


def cpu_cores(quantity: str) -> float:
    return int(quantity[:-1]) / 1000 if quantity.endswith("m") else float(quantity)


def memory_bytes(quantity: str) -> int:
    number, unit = re.fullmatch(r"([0-9.]+)([A-Za-z]*)", quantity).groups()
    return int(float(number) * UNITS.get(unit, 1))


def qos_class(containers: list[dict]) -> str:
    anything_set, guaranteed = False, True
    for container in containers:
        requests, limits = container.get("requests", {}), container.get("limits", {})
        for resource in ("cpu", "memory"):
            anything_set |= resource in requests or resource in limits
            # Kubernetes copies a limit into a missing request, so limit-only counts as equal.
            request = requests.get(resource, limits.get(resource))
            if resource not in limits or request != limits[resource]:
                guaranteed = False
    if not anything_set:
        return "BestEffort"
    return "Guaranteed" if guaranteed else "Burstable"


assert cpu_cores("500m") == 0.5 and cpu_cores("2") == 2.0
assert memory_bytes("16Gi") == 16 * 2**30 and memory_bytes("1G") == 10**9
assert qos_class([{"requests": {"cpu": "2", "memory": "8Gi"}, "limits": {"cpu": "2", "memory": "8Gi"}}]) == "Guaranteed"
assert qos_class([{"limits": {"cpu": "1", "memory": "1Gi"}}]) == "Guaranteed"
assert qos_class([{"requests": {"cpu": "2", "memory": "8Gi"}, "limits": {"memory": "8Gi"}}]) == "Burstable"
assert qos_class([{}]) == "BestEffort"
print("quantities parsed and QoS classes match the Kubernetes rules")

### Detecting CPU throttling

cAdvisor exposes `container_cpu_cfs_periods_total` and `container_cpu_cfs_throttled_periods_total`. A throttled fraction above roughly 25% means the CPU limit is too low for the workload's bursts; Spark tasks and dbt runs then take longer while the node may be idle.

In [ ]:
THROTTLE_THRESHOLD = 0.25
samples = {"api": (48_000, 1_200), "spark-executor": (200_000, 76_000), "dbt": (12_000, 5_000)}

flagged = [name for name, (periods, throttled) in samples.items() if throttled / periods > THROTTLE_THRESHOLD]
for name, (periods, throttled) in samples.items():
    print(f"{name:<15} {throttled / periods:6.1%}")
assert flagged == ["spark-executor", "dbt"]

### Sizing JVM containers

Size the memory limit from the JVM, not the other way round. Two common approaches: let the JVM take a percentage of the container (`-XX:MaxRAMPercentage=75`), or set the heap explicitly and add overhead (Spark on Kubernetes requests `spark.executor.memory + spark.executor.memoryOverhead`, where the overhead defaults to `max(384 MiB, 10%)`, raised to 40% for non-JVM jobs via `spark.kubernetes.memoryOverheadFactor`).

In [ ]:
import math

HEAP_GIB = 12
MAX_RAM_PERCENTAGE = 0.75
OVERHEAD_FLOOR_MIB = 384
OVERHEAD_FRACTION = 0.10

limit_with_max_ram = math.ceil(HEAP_GIB / MAX_RAM_PERCENTAGE)
overhead_mib = max(OVERHEAD_FLOOR_MIB, int(HEAP_GIB * 1024 * OVERHEAD_FRACTION))
limit_with_overhead = math.ceil((HEAP_GIB * 1024 + overhead_mib) / 1024)
print(f"MaxRAMPercentage=75 -> {limit_with_max_ram} GiB limit; heap + {overhead_mib} MiB overhead -> {limit_with_overhead} GiB")
assert (limit_with_max_ram, overhead_mib, limit_with_overhead) == (16, 1228, 14)

## Scheduling: node selectors, affinity, taints and tolerations

The scheduler filters nodes that can run the pod, then scores the survivors. The filters a data engineer controls:

+ **`nodeSelector`**: the node must carry these labels (`node-pool: spark`).
+ **Node affinity**: richer rules, `requiredDuringSchedulingIgnoredDuringExecution` (hard) or `preferred...` (soft, weighted), with operators `In`, `NotIn`, `Exists`.
+ **Pod affinity / anti-affinity**: co-locate with or spread away from pods with given labels, per topology key (node, zone). **Topology spread constraints** spread replicas evenly across zones.
+ **Taints and tolerations**: a taint on a node (`dedicated=spark:NoSchedule`) repels every pod that does not tolerate it. Taints keep general pods *off* special nodes (GPU, spot); a toleration only permits scheduling there, so pair it with a selector or affinity to *attract* the pod. `NoExecute` also evicts running pods.

```yaml
spec:
  nodeSelector: {node-pool: spark-spot}
  tolerations:
    - {key: "dedicated", operator: "Equal", value: "spark", effect: "NoSchedule"}
    - {key: "cloud.google.com/gke-spot", operator: "Equal", value: "true", effect: "NoSchedule"}
  affinity:
    podAntiAffinity:
      preferredDuringSchedulingIgnoredDuringExecution:
        - weight: 100
          podAffinityTerm:
            labelSelector: {matchLabels: {app: trino-worker}}
            topologyKey: kubernetes.io/hostname
```

A typical node-pool layout: a small on-demand pool for system components, Airflow schedulers and Spark drivers; a tainted spot pool for executors and retryable batch pods; a high-memory pool for wide joins. Run drivers on on-demand nodes: losing a driver loses the whole application, while losing an executor only retries its tasks.

The cell below is a toy scheduler with the real filter order and a least-allocated score. It also explains why a pod stays `Pending`, which is exactly the information `kubectl describe pod` shows in its events.

In [ ]:
NODES = [
    {"name": "system-1", "labels": {"node-pool": "system"}, "taints": [], "cpu": 4, "mem_gi": 16,
     "used_cpu": 3.0, "used_mem_gi": 10},
    {"name": "spot-1", "labels": {"node-pool": "spark-spot"}, "taints": [("dedicated", "spark", "NoSchedule")],
     "cpu": 16, "mem_gi": 64, "used_cpu": 12, "used_mem_gi": 50},
    {"name": "spot-2", "labels": {"node-pool": "spark-spot"}, "taints": [("dedicated", "spark", "NoSchedule")],
     "cpu": 16, "mem_gi": 64, "used_cpu": 4, "used_mem_gi": 16},
]


def schedule(pod: dict, nodes: list[dict]) -> tuple[str | None, dict[str, str]]:
    reasons, feasible = {}, []
    for node in nodes:
        if any(node["labels"].get(k) != v for k, v in pod.get("node_selector", {}).items()):
            reasons[node["name"]] = "node(s) didn't match node selector"
        elif any(taint not in pod.get("tolerations", []) for taint in node["taints"] if taint[2] == "NoSchedule"):
            reasons[node["name"]] = "node(s) had untolerated taint"
        elif node["cpu"] - node["used_cpu"] < pod["cpu"]:
            reasons[node["name"]] = "Insufficient cpu"
        elif node["mem_gi"] - node["used_mem_gi"] < pod["mem_gi"]:
            reasons[node["name"]] = "Insufficient memory"
        else:
            feasible.append(node)

    def free_fraction(node: dict) -> float:  # LeastAllocated scoring: prefer emptier nodes
        cpu_free = (node["cpu"] - node["used_cpu"] - pod["cpu"]) / node["cpu"]
        mem_free = (node["mem_gi"] - node["used_mem_gi"] - pod["mem_gi"]) / node["mem_gi"]
        return (cpu_free + mem_free) / 2

    best = max(feasible, key=free_fraction)["name"] if feasible else None
    return best, reasons


executor = {"cpu": 4, "mem_gi": 18, "node_selector": {"node-pool": "spark-spot"},
            "tolerations": [("dedicated", "spark", "NoSchedule")]}
node, reasons = schedule(executor, NODES)
print("executor ->", node, reasons)
assert node == "spot-2" and reasons["spot-1"] == "Insufficient memory"

no_toleration = {**executor, "tolerations": []}
node, reasons = schedule(no_toleration, NODES)
print("without toleration ->", node, reasons)
assert node is None and reasons["spot-1"] == reasons["spot-2"] == "node(s) had untolerated taint"

## Pod lifecycle: init containers and probes

**Init containers** run to completion, in order, before the app containers start. Uses: wait for a database, run migrations, download a JAR or model artifact into a shared `emptyDir`, fix volume permissions. **Sidecars** (native since 1.29 as init containers with `restartPolicy: Always`) run alongside: log shippers, the Cloud SQL proxy, git-sync for Airflow DAGs.

| Probe | Failure means | Typical mistake |
|---|---|---|
| `startupProbe` | Still starting; other probes wait until it succeeds | Missing on slow-starting JVMs, so liveness kills them during startup |
| `readinessProbe` | Remove the pod from Service endpoints, do not restart | Checking downstream dependencies, so one database blip removes every pod |
| `livenessProbe` | Restart the container | Too aggressive thresholds causing restart loops under load |

```yaml
initContainers:
  - name: wait-for-postgres
    image: postgres:16
    command: ["sh", "-c", "until pg_isready -h postgres -p 5432; do sleep 2; done"]
containers:
  - name: trino
    image: trinodb/trino:463
    startupProbe:   {httpGet: {path: /v1/info, port: 8080}, failureThreshold: 30, periodSeconds: 10}
    readinessProbe: {httpGet: {path: /v1/info, port: 8080}, periodSeconds: 10}
    livenessProbe:  {httpGet: {path: /v1/info, port: 8080}, periodSeconds: 20, failureThreshold: 3}
```

On termination the pod is removed from endpoints, a `preStop` hook runs, the container gets `SIGTERM`, and after `terminationGracePeriodSeconds` (default 30) it gets `SIGKILL`. Long-running tasks need a longer grace period and a process that handles `SIGTERM` (exec-form entrypoint).

## Storage: volumes, PersistentVolumes and StorageClasses

+ `emptyDir`: scratch space for the pod's lifetime (Spark shuffle spill; `medium: Memory` for tmpfs, which counts against memory).
+ `configMap` / `secret` volumes: configuration files.
+ **PersistentVolumeClaim (PVC)**: a request for storage (size, access mode). A **StorageClass** names a provisioner and parameters (`gp3`, `pd-ssd`) and dynamically creates the **PersistentVolume (PV)**. `volumeBindingMode: WaitForFirstConsumer` provisions the disk in the zone where the pod is scheduled, avoiding zone mismatches. `reclaimPolicy: Retain` keeps the disk when the claim is deleted.
+ Access modes: `ReadWriteOnce` (one node; block disks), `ReadOnlyMany`, `ReadWriteMany` (shared file systems such as EFS or Filestore), `ReadWriteOncePod`.

For data platforms, keep durable data in object storage (S3, GCS) and use PVCs for state that must be local: Kafka logs, ZooKeeper, a Postgres metadata DB if not managed, Airflow logs if not shipped to object storage.

```yaml
apiVersion: storage.k8s.io/v1
kind: StorageClass
metadata: {name: fast-ssd}
provisioner: ebs.csi.aws.com
parameters: {type: gp3, iops: "6000"}
volumeBindingMode: WaitForFirstConsumer
reclaimPolicy: Retain
allowVolumeExpansion: true
```

## Networking: Services, Ingress and load balancing

A **Service** gives a stable virtual IP and DNS name (`trino.analytics.svc.cluster.local`) in front of the pods matching its selector; kube-proxy load-balances connections across ready endpoints.

| Type | Reachable from | Notes |
|---|---|---|
| ClusterIP (default) | Inside the cluster | Service-to-service traffic |
| Headless (`clusterIP: None`) | DNS returns pod IPs | StatefulSets: `kafka-0.kafka.data.svc` |
| NodePort | Every node's IP on a port (30000-32767) | Basic external access, mostly for testing |
| LoadBalancer | A cloud load balancer | One LB per Service; costs add up |
| ExternalName | DNS CNAME to an external host | Alias for an external database |

To make an internal ClusterIP service external, change `type: ClusterIP` to `type: LoadBalancer` (or NodePort), or put it behind an Ingress. Port chain: the container listens on `containerPort`, the Service maps its `port` to `targetPort`, and an Ingress routes host and path to the Service port; the browser uses 80/443 on the Ingress. The `selector` must match the pod labels exactly, otherwise the Service has no endpoints.

```yaml
apiVersion: v1
kind: Service
metadata: {name: superset, namespace: bi}
spec:
  type: ClusterIP
  selector: {app: superset}
  ports: [{name: http, port: 8080, targetPort: 8088}]
---
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: superset
  namespace: bi
  annotations: {cert-manager.io/cluster-issuer: letsencrypt}
spec:
  ingressClassName: nginx
  tls: [{hosts: [bi.acme.com], secretName: superset-tls}]
  defaultBackend: {service: {name: fallback, port: {number: 80}}}
  rules:
    - host: bi.acme.com
      http:
        paths:
          - {path: /, pathType: Prefix, backend: {service: {name: superset, port: {number: 8080}}}}
```

An **Ingress** is layer-7 routing (host and path) implemented by an Ingress controller (NGINX, Traefik, cloud controllers); one load balancer serves many Services. TLS is a `kubernetes.io/tls` Secret referenced under `tls:`, usually issued by cert-manager. The **default backend** handles requests that match no rule. The Gateway API is the newer, more expressive successor. Services in another namespace are reached by `<service>.<namespace>.svc.cluster.local`.

### Where you'll use it

- **Stateful services:** a Kafka broker or a database in Kubernetes needs a PersistentVolume so its data survives a pod restart.

In [ ]:
import yaml

MANIFESTS = list(yaml.safe_load_all("""
apiVersion: v1
kind: Service
metadata: {name: superset, namespace: bi}
spec:
  selector: {app: superset}
  ports: [{port: 8080, targetPort: 8088}]
---
apiVersion: v1
kind: Pod
metadata: {name: superset-7d9f, namespace: bi, labels: {app: superset, tier: web}}
spec:
  containers: [{name: superset, image: apache/superset:4.1.1, ports: [{containerPort: 8088}]}]
---
apiVersion: v1
kind: Pod
metadata: {name: superset-worker-1, namespace: bi, labels: {app: superset-worker}}
spec:
  containers: [{name: worker, image: apache/superset:4.1.1}]
"""))


def endpoints(service: dict, pods: list[dict]) -> list[str]:
    selector = service["spec"]["selector"]
    return [
        pod["metadata"]["name"] for pod in pods
        if pod["metadata"]["namespace"] == service["metadata"]["namespace"]
        and all(pod["metadata"]["labels"].get(k) == v for k, v in selector.items())
    ]


service = next(m for m in MANIFESTS if m["kind"] == "Service")
pods = [m for m in MANIFESTS if m["kind"] == "Pod"]
selected = endpoints(service, pods)
target_port = service["spec"]["ports"][0]["targetPort"]
container_ports = {p["containerPort"] for pod in pods if pod["metadata"]["name"] in selected
                   for c in pod["spec"]["containers"] for p in c.get("ports", [])}
print(f"{service['metadata']['name']}.{service['metadata']['namespace']}.svc.cluster.local -> {selected}")
assert selected == ["superset-7d9f"] and target_port in container_ports

## Namespaces, configuration and secrets

**Namespaces** scope names, RBAC, quotas, limit ranges and network policies. Putting everything in `default` means no isolation, no per-team quotas, and one mistaken `kubectl delete --all` away from an outage. Typical layout: one namespace per team or per environment (`airflow`, `spark-jobs`, `bi`).

**ConfigMaps** hold non-secret configuration; **Secrets** hold credentials. Secrets are only base64-encoded, so protect them: enable encryption at rest for etcd, restrict `get secret` with RBAC, and prefer an external source of truth (External Secrets Operator syncing from AWS Secrets Manager or Vault, or the Secrets Store CSI driver). Mount secrets as files rather than environment variables where possible, and use workload identity (IRSA on EKS, Workload Identity on GKE) so pods reach cloud storage without static keys at all.

```yaml
apiVersion: external-secrets.io/v1beta1
kind: ExternalSecret
metadata: {name: warehouse-creds, namespace: airflow}
spec:
  refreshInterval: 1h
  secretStoreRef: {name: aws-secrets-manager, kind: ClusterSecretStore}
  target: {name: warehouse-creds}
  data:
    - {secretKey: password, remoteRef: {key: prod/warehouse, property: password}}
```

## Security

+ **RBAC**: Roles/ClusterRoles grant verbs on resources; RoleBindings bind them to users, groups or ServiceAccounts. Give each workload its own ServiceAccount with least privilege (the Spark driver needs to create and delete executor pods in its namespace, nothing more).
+ **Pod Security Standards** (`restricted` profile via Pod Security Admission): non-root, no privilege escalation, dropped capabilities, read-only root filesystem, seccomp `RuntimeDefault`.
+ **NetworkPolicies**: default-deny ingress per namespace, then allow only needed flows (requires a CNI that enforces them).
+ **Supply chain**: scanned, signed images from a private registry; admission policies (Kyverno, OPA Gatekeeper) to block `latest` tags, privileged pods or missing resource requests.
+ Audit logging on the API server; keep the cluster and nodes patched; private control-plane endpoints.

```yaml
securityContext:
  runAsNonRoot: true
  runAsUser: 10001
  allowPrivilegeEscalation: false
  readOnlyRootFilesystem: true
  capabilities: {drop: ["ALL"]}
  seccompProfile: {type: RuntimeDefault}
```

## Batch jobs and CronJobs

```yaml
apiVersion: batch/v1
kind: CronJob
metadata: {name: nightly-dbt, namespace: analytics}
spec:
  schedule: "0 2 * * *"
  timeZone: "Etc/UTC"
  concurrencyPolicy: Forbid
  startingDeadlineSeconds: 3600
  successfulJobsHistoryLimit: 3
  failedJobsHistoryLimit: 5
  jobTemplate:
    spec:
      backoffLimit: 2
      activeDeadlineSeconds: 7200
      ttlSecondsAfterFinished: 86400
      template:
        spec:
          restartPolicy: Never
          serviceAccountName: dbt-runner
          containers:
            - name: dbt
              image: ghcr.io/acme/dbt:3f2a1c9
              args: ["build", "--target", "prod"]
              resources:
                requests: {cpu: "1", memory: 2Gi}
                limits: {memory: 2Gi}
```

+ `concurrencyPolicy`: `Allow` (overlapping runs, dangerous for non-idempotent loads), `Forbid` (skip the new run while one is active), `Replace` (kill the running one).
+ `startingDeadlineSeconds`: how late a missed run may still start (after controller downtime); without it, more than 100 missed schedules stop the CronJob.
+ `backoffLimit` retries failed pods; `activeDeadlineSeconds` kills runaway jobs; `ttlSecondsAfterFinished` cleans up.
+ Jobs must be idempotent: a pod can be retried after partially writing output, and CronJobs can occasionally fire twice.

CronJobs are fine for independent scheduled tasks; use Airflow (or Argo Workflows) when you need dependencies, backfills, and lineage across tasks.

### Where you'll use it

- **Nightly loads:** a CronJob starts a Python extract at 02:00 and retries it if the source is down.
- **Backfills:** a Job with a fixed number of parallel workers processes a month of partitions.

In [ ]:
SCHEDULE_HOURS = [2, 3, 4, 5]  # hourly runs from 02:00
RUN_HOURS = 2.5                # each run takes longer than the interval


def simulate(policy: str) -> list[str]:
    events, active = [], []  # active: list of (start, end)
    for start in SCHEDULE_HOURS:
        active = [(s, e) for s, e in active if e > start]
        if active and policy == "Forbid":
            events.append(f"{start:02d}:00 skipped")
            continue
        if active and policy == "Replace":
            events.append(f"{start:02d}:00 killed run from {active[0][0]:02d}:00")
            active = []
        active.append((start, start + RUN_HOURS))
        events.append(f"{start:02d}:00 started ({len(active)} running)")
    return events


for policy in ["Allow", "Forbid", "Replace"]:
    print(policy, simulate(policy))
assert max(int(e.split("(")[1][0]) for e in simulate("Allow") if "started" in e) == 3
assert sum("skipped" in e for e in simulate("Forbid")) == 2

## Autoscaling

**Horizontal Pod Autoscaler (HPA)** sets replicas to `ceil(current * currentMetric / target)`, ignoring changes within a 10% tolerance, bounded by min and max replicas. It scales on CPU, memory or custom and external metrics; for queue-driven workers (Celery, Kafka consumers) scale on queue depth or consumer lag with **KEDA**, which can also scale to zero. `behavior` stabilization windows prevent flapping (scale-down waits 5 minutes by default).

**Vertical Pod Autoscaler (VPA)** recommends or applies requests from observed usage; use it in recommendation mode to right-size batch jobs. Do not let HPA and VPA act on the same CPU metric.

**Cluster Autoscaler / Karpenter** add nodes when pods are `Pending` for lack of capacity and remove underused nodes whose pods can move elsewhere. Pods with local storage, restrictive PDBs or `cluster-autoscaler.kubernetes.io/safe-to-evict: "false"` block scale-down.

### Where you'll use it

- **Bursty workloads:** a stream consumer scales out when lag grows and back in when the topic is quiet.

In [ ]:
TOLERANCE = 0.1


def hpa_desired(current: int, utilization: float, target: float, min_replicas: int = 1, max_replicas: int = 30) -> int:
    ratio = utilization / target
    if abs(ratio - 1.0) <= TOLERANCE:
        return current
    return max(min_replicas, min(max_replicas, math.ceil(current * ratio)))


def nodes_needed(pods: int, pod_cpu: float, pod_mem_gi: float, node_cpu: float, node_mem_gi: float,
                 headroom: float = 0.1) -> int:
    usable_cpu, usable_mem = node_cpu * (1 - headroom), node_mem_gi * (1 - headroom)
    per_node = min(int(usable_cpu // pod_cpu), int(usable_mem // pod_mem_gi))
    return math.ceil(pods / per_node)


print("4 replicas at 90% CPU, target 60% ->", hpa_desired(4, 90, 60))
print("10 replicas at 63%, target 60% ->", hpa_desired(10, 63, 60), "(within tolerance)")
print("40 executors of 3 CPU / 18 GiB on 16 CPU / 64 GiB nodes ->", nodes_needed(40, 3, 18, 16, 64), "nodes")
assert hpa_desired(4, 90, 60) == 6 and hpa_desired(10, 63, 60) == 10 and hpa_desired(2, 0, 60) == 1
assert nodes_needed(40, 3, 18, 16, 64) == 14  # memory, not CPU, is the binding constraint

## Node maintenance and Pod Disruption Budgets

To service a node: `kubectl cordon <node>` (no new pods), `kubectl drain <node> --ignore-daemonsets --delete-emptydir-data` (evict pods respecting PDBs), do the work, then `kubectl uncordon <node>`.

A **PodDisruptionBudget** limits *voluntary* disruptions (drains, upgrades, autoscaler scale-down), not crashes or node failures: `minAvailable: 2` or `maxUnavailable: 1` for pods matching a selector. Use them for quorum systems (Kafka, ZooKeeper) and serving tiers; a PDB that allows zero disruptions blocks drains and cluster upgrades forever.

Under node memory or disk pressure, the kubelet evicts pods on its own: BestEffort first, then Burstable pods using the most memory above their requests, Guaranteed last.

In [ ]:
def evictions_allowed(healthy: int, min_available: int | None = None, max_unavailable: int | None = None,
                      desired: int | None = None) -> int:
    if min_available is not None:
        return max(0, healthy - min_available)
    return max(0, max_unavailable - ((desired or healthy) - healthy))


def eviction_order(pods: list[dict]) -> list[str]:
    rank = {"BestEffort": 0, "Burstable": 1, "Guaranteed": 2}
    return [p["name"] for p in sorted(pods, key=lambda p: (rank[p["qos"]], -(p["usage_gi"] - p["request_gi"])))]


assert evictions_allowed(healthy=3, min_available=2) == 1
assert evictions_allowed(healthy=2, max_unavailable=1, desired=3) == 0  # one replica already down: drain blocks
order = eviction_order([
    {"name": "trino-worker", "qos": "Guaranteed", "usage_gi": 15, "request_gi": 16},
    {"name": "adhoc-notebook", "qos": "BestEffort", "usage_gi": 3, "request_gi": 0},
    {"name": "airflow-task-a", "qos": "Burstable", "usage_gi": 6, "request_gi": 2},
    {"name": "airflow-task-b", "qos": "Burstable", "usage_gi": 3, "request_gi": 2},
])
print("eviction order under memory pressure:", order)
assert order == ["adhoc-notebook", "airflow-task-a", "airflow-task-b", "trino-worker"]

## Running data workloads

### Spark on Kubernetes

`spark-submit --master k8s://https://<api-server>` creates a driver pod, which creates executor pods through the API (so it needs a ServiceAccount with pod create/delete rights). The **Spark Operator** (Kubeflow) wraps this in a `SparkApplication` custom resource that Airflow or Argo can submit and watch.

```yaml
apiVersion: sparkoperator.k8s.io/v1beta2
kind: SparkApplication
metadata: {name: daily-orders, namespace: spark-jobs}
spec:
  type: Python
  mode: cluster
  image: ghcr.io/acme/spark-jobs:3f2a1c9
  mainApplicationFile: local:///app/jobs/daily_orders.py
  sparkVersion: "3.5.3"
  sparkConf:
    spark.dynamicAllocation.enabled: "true"
    spark.dynamicAllocation.shuffleTracking.enabled: "true"
    spark.kubernetes.executor.podTemplateFile: /templates/spot-executor.yaml
    spark.decommission.enabled: "true"
  driver: {cores: 1, memory: 4g, serviceAccount: spark-driver, nodeSelector: {node-pool: on-demand}}
  executor: {instances: 10, cores: 4, memory: 14g, memoryOverhead: 2g}
```

Points interviewers probe: there is no external shuffle service, so dynamic allocation relies on shuffle tracking; executor decommissioning migrates shuffle blocks when spot nodes are reclaimed; local SSDs or `emptyDir` serve as spill space; executors per node is bounded by memory more often than CPU; and logs and the Spark History Server need event logs in object storage because pods disappear.

### Batch schedulers: Kueue, Volcano and Apache YuniKorn

The default kube-scheduler places pods one at a time and has no idea that 50 executor pods belong to one Spark job. Two problems follow on a busy shared cluster: **partial starts** (a driver and 10 of 50 executors start, hold resources, and wait forever for the rest, possibly deadlocking with another job doing the same) and **no fair sharing** between teams beyond static ResourceQuotas. Batch schedulers add job-level awareness.

| Tool | What it adds | How it plugs in |
|---|---|---|
| **Kueue** (Kubernetes SIG project) | job queueing and admission: a job waits until its whole quota fits, with ClusterQueues, LocalQueues, ResourceFlavors, cohorts for sharing quota between queues, and preemption | does not replace kube-scheduler; it decides *when* a job is admitted. Supports Job, JobSet, RayJob, RayCluster and plain pods (SparkApplication support is alpha) |
| **Volcano** | gang scheduling through PodGroups (all pods start or none do), queues, fair share | a separate scheduler; Spark supports it since 3.3 via `spark.kubernetes.scheduler.name=volcano` and a PodGroup template |
| **Apache YuniKorn** | hierarchical queues with guaranteed and maximum resources, gang scheduling (with a soft fallback), application-aware sorting | replaces the default scheduler for the pods it owns; Spark pods select it with `spark.kubernetes.scheduler.name=yunikorn` and a queue label |

```bash
# Spark on Kubernetes with YuniKorn: the job lands in a team queue and is scheduled as one application.
spark-submit --master k8s://https://<api-server> --deploy-mode cluster \
  --conf spark.kubernetes.scheduler.name=yunikorn \
  --conf spark.kubernetes.driver.label.queue=root.analytics \
  --conf spark.kubernetes.executor.label.queue=root.analytics \
  --conf spark.kubernetes.driver.annotation.yunikorn.apache.org/app-id={{APP_ID}} \
  --conf spark.kubernetes.executor.annotation.yunikorn.apache.org/app-id={{APP_ID}} \
  local:///app/jobs/daily_orders.py
```

How to choose: start with **Kueue** when the problem is quota and queueing across teams for standard Job-style workloads (including Ray and ML training). Choose **YuniKorn** or **Volcano** when Spark or MPI-style jobs need gang scheduling and hierarchical queues; YuniKorn is the common pick for Spark-heavy platforms, Volcano for mixed AI and batch frameworks. Kueue and a gang scheduler are often combined, with Kueue doing admission and the scheduler doing atomic placement.

### Airflow on Kubernetes

+ **KubernetesExecutor**: one pod per task, created by the scheduler. Strong isolation and per-task resources (`executor_config` / pod override), zero idle cost, but pod startup adds seconds to every task.
+ **CeleryExecutor** on Kubernetes: long-running workers (Deployment or StatefulSet) pull from a Redis or RabbitMQ queue; fast task start, scale with KEDA on queue depth.
+ **CeleryKubernetesExecutor**: both, chosen per task queue.
+ **KubernetesPodOperator**: run any image as a task, regardless of executor; the standard way to run dbt, Spark-submit or non-Python jobs with their own dependencies.

Deploy with the official Helm chart; ship DAGs with git-sync or bake them into the image; send task logs to object storage; give each team its own namespace and ServiceAccount for pod-operator tasks.

### Operators

An **Operator** is a controller plus Custom Resource Definitions that encode operational knowledge for one system: install, upgrade, scale, fail over, back up. Instead of hand-managing a StatefulSet for Kafka, you declare a `Kafka` resource (Strimzi) and the operator reconciles brokers, topics and users. Data-platform examples: Strimzi (Kafka), Flink Kubernetes Operator, Spark Operator, CloudNativePG and Zalando Postgres, Trino and Superset charts. Operators are worth it when the system has non-trivial day-2 operations that a plain Deployment cannot express.

## Observability and debugging

**Metrics**: Prometheus scraping kubelet/cAdvisor (container CPU, memory, throttling), kube-state-metrics (desired versus actual replicas, pod phases, restarts) and application endpoints; Grafana dashboards; alerts on restarts, OOM kills, Pending pods and Job failures. **Logs**: containers write to stdout and stderr; a DaemonSet (Fluent Bit, Vector) ships node log files to a central store (Loki, Elasticsearch, Cloud Logging). `kubectl logs` only shows what is still on the node, and `--previous` shows the last crashed container.

| Symptom | What to check |
|---|---|
| `Pending` | `kubectl describe pod` events: insufficient CPU or memory, untolerated taint, node selector or affinity mismatch, unbound PVC, quota exceeded |
| `ImagePullBackOff` | Image name or tag, registry credentials (`imagePullSecrets`), architecture |
| `CrashLoopBackOff` | `kubectl logs --previous`; bad config, missing secret, failing liveness probe |
| `OOMKilled` (exit 137) | Memory limit below real peak; JVM overhead or Python workers not counted |
| Slow but running | CPU throttling, noisy neighbours, data skew in Spark, network or storage throughput |
| `Evicted` | Node memory or ephemeral-storage pressure; set requests and `ephemeral-storage` limits |

```bash
kubectl get pods -n spark-jobs -o wide
kubectl describe pod daily-orders-driver -n spark-jobs      # events explain scheduling and probe failures
kubectl logs daily-orders-exec-3 -n spark-jobs --previous
kubectl top pod -n spark-jobs --containers
kubectl get events -n spark-jobs --sort-by=.lastTimestamp
kubectl debug -it daily-orders-driver --image=busybox --target=spark-kubernetes-driver
```

## Docker Swarm versus Kubernetes

Swarm is simpler but has a small ecosystem and little scheduling or autoscaling depth; for data platforms (Spark, Flink, Airflow, Strimzi, batch schedulers) Kubernetes is the default. See also [Docker](../docker/docker.ipynb).

## Practice scenarios

+ **Spark batch platform on EKS**: on-demand pool for drivers, tainted spot pool for executors, Karpenter for node scaling, YuniKorn queues per team with gang scheduling, decommissioning enabled, event logs in S3.
+ **Multi-tenant Airflow**: namespace and ServiceAccount per team, ResourceQuotas, KubernetesPodOperator tasks with per-task resources, NetworkPolicies between teams.
+ **Streaming ingestion**: Strimzi Kafka with PDBs and rack awareness across zones, Flink jobs managed by the Flink operator with checkpoints in object storage.
+ **Self-service dbt**: CronJobs or Airflow tasks running a pinned dbt image with `concurrencyPolicy: Forbid` and workload identity to the warehouse.

Questions to check yourself:

1. A pod is `Pending` with "0/3 nodes are available: 3 node(s) had untolerated taint". What do you change?
2. Why can a CPU limit slow a job on an idle node, while a memory limit kills it?
3. Why should Spark drivers avoid spot nodes while executors can use them?
4. What does a PDB protect against, and what does it not?

### Where you'll use it

- **Operator-managed systems:** Strimzi (Kafka), Spark Operator and the Flink operator turn a cluster into a few custom resources.